# Tree-type network of small neural networks - Colab runner
**Runtime -> Change runtime type -> T4 GPU** (or better) before running.
Order: setup -> unit tests -> quick trial (~10-15 min, checks everything works) -> full runs -> report.
Results are copied to Google Drive after each stage so a disconnect does not lose them.

In [ ]:
import torch, subprocess
print(torch.__version__, 'cuda:', torch.cuda.is_available())
print(subprocess.run(['nvidia-smi','-L'],capture_output=True,text=True).stdout)

## 1. Get the code
Either upload `tree_nn_project.zip` (simplest) or clone your private repo with a personal access token.

In [ ]:
from google.colab import files
up = files.upload()            # choose tree_nn_project.zip
!unzip -q -o tree_nn_project.zip -d /content
%cd /content/tree_nn_project

In [ ]:
# ALTERNATIVE: private GitHub repo (token needs 'repo' scope; do not share this notebook with the token in it)
# TOKEN='ghp_...'; USER='<github-user>'; REPO='<repo>'
# !git clone https://{TOKEN}@github.com/{USER}/{REPO}.git /content/tree_nn_project
# %cd /content/tree_nn_project

In [ ]:
!pip -q install reportlab scikit-learn matplotlib pillow
from google.colab import drive
drive.mount('/content/drive')
SAVE = '/content/drive/MyDrive/tree_nn_results'
!mkdir -p {SAVE}

## 2. Unit tests of the tree logic (CPU, ~1 min)

In [ ]:
!python tests/test_tree_logic.py

## 3. Quick trial - run this BEFORE the long jobs
Small subset, few widths, shallow tree. If anything crashes here, copy the traceback back to Claude.

In [ ]:
!python scripts/run_tree.py --dataset cifar10 --n_train 5000 --trunk scratch_cnn --trunk_epochs 2 \
    --widths 0,16 --steps 500 --max_depth 4 --max_nodes 300 --out results/trial_cifar10
!python scripts/minimize.py --run results/trial_cifar10
!python scripts/visualize_rf.py --run results/trial_cifar10 --n_children 4 --max_depth 2

## 4. Main runs: interpolation (clean labels and 15% label noise)
`--max_nodes` caps the tree size; if the cap is hit the tree may stop short of 100% - check `tree_train_acc` and say so in your report.
Adjust `--trunk_epochs` (more epochs = stronger trunk = smaller tree) and `--widths` as you like.

In [ ]:
!rm -rf results/trial_cifar10
!python scripts/run_tree.py --dataset cifar10 --trunk scratch_cnn --trunk_epochs 3 --out results/cifar10_clean
!python scripts/run_tree.py --dataset cifar10 --trunk scratch_cnn --trunk_epochs 3 --noise 0.15 --out results/cifar10_noise15
!cp -r results {SAVE}/

In [ ]:
!python scripts/run_tree.py --dataset catsdogs --img_size 64 --trunk scratch_cnn --trunk_epochs 3 --out results/catsdogs_clean
!python scripts/run_tree.py --dataset catsdogs --img_size 64 --trunk scratch_cnn --trunk_epochs 3 --noise 0.15 --out results/catsdogs_noise15
!cp -r results {SAVE}/

## 5. Size minimisation and receptive fields

In [ ]:
for t in ['cifar10_clean','cifar10_noise15','catsdogs_clean','catsdogs_noise15']:
    !python scripts/minimize.py --run results/{t}
for t,d in [('cifar10_clean','cifar10'),('cifar10_noise15','cifar10'),('catsdogs_clean','catsdogs'),('catsdogs_noise15','catsdogs')]:
    !python scripts/visualize_rf.py --run results/{t} --dataset {d}
!cp -r results {SAVE}/

## 6. Double descent (longest part)
Model-wise / epoch-wise with width-k ResNet-18 and label noise, then the tree-wise sweep.
`double_descent.py` and `tree_sweep.py` resume from their json if the session is interrupted (re-run the cell).
Use `--widths 2,4,8,16,32` and `--epochs 60` for a faster first look.

In [ ]:
!python scripts/double_descent.py --dataset cifar10 --noise 0.15 --n_train 15000 --out results/dd_cifar10
!cp -r results {SAVE}/

In [ ]:
!python scripts/tree_sweep.py --dataset cifar10 --noise 0.15 --n_train 15000 --out results/sweep_cifar10
!cp -r results {SAVE}/

## 7. Report
1. Look at the figures/tables in `results/`, then write `interpretation.md` (start from `interpretation_template.md`).
2. Push the code to a PRIVATE GitHub repo and put its URL below.

In [ ]:
REPO_URL = 'https://github.com/<you>/<private-repo>'   # <- edit
!python make_report.py --results results --repo_url {REPO_URL} --interpretation interpretation.md --out report.pdf
from google.colab import files; files.download('report.pdf')
!cp report.pdf {SAVE}/